In [ ]:
import rioxarray as rxr
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
output_root = Path("/explore/nobackup/people/ajkerr1/lfm_c8_1_wac_37924350/explicit_aoi_api")
chips = sorted((output_root / "chips").glob("*.tif"))
labels = sorted((output_root / "labels").glob("*.npz"))

In [ ]:
chip_by_id = {p.stem.removesuffix("_input_wac_chip"): p for p in chips}
label_by_id = {p.stem.removesuffix("_label"): p for p in labels}
assert chip_by_id.keys() == label_by_id.keys()
pairs = [(sample_id, chip_by_id[sample_id], label_by_id[sample_id]) for sample_id in chip_by_id]

In [ ]:
reference_dir = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/7_band_vis_uv/inst_seg/chips")

def chip_sample_id(path):
    stem = path.stem
    for suffix in ("_input_wac_static_chip", "_input_wac_chip"):
        if stem.endswith(suffix):
            return stem.removesuffix(suffix)
    return stem

reference_by_id = {chip_sample_id(path): path for path in reference_dir.glob("*.tif")}
assert set(chip_by_id) <= set(reference_by_id)

def load_first_vis(path):
    raster = rxr.open_rasterio(path, masked=True)
    names = raster.attrs.get("long_name", tuple(f"band {i}" for i in range(raster.shape[0])))
    names = (names,) if isinstance(names, str) else names
    band = next((i for i, name in enumerate(names) if "vis" in str(name).lower()), 0)
    image = raster.values[band]
    raster.close()
    return image, names[band]

fig, axes = plt.subplots(4, len(pairs), figsize=(4 * len(pairs), 13), squeeze=False)

for col, (sample_id, chip, label) in enumerate(pairs):
    image, band_name = load_first_vis(chip)
    reference, reference_band_name = load_first_vis(reference_by_id[sample_id])

    with np.load(label, allow_pickle=False) as archive:
        mask = archive["mask"]
        count = int(archive["num_craters"])
    present = set(np.unique(mask[mask > 0]))
    missing = sorted(set(range(1, count + 1)) - present)
    instances = np.ma.masked_where(mask == 0, (mask - 1) % 20)
    combined = np.concatenate((image[np.isfinite(image)], reference[np.isfinite(reference)]))
    vmin, vmax = np.percentile(combined, (2, 98))

    axes[0, col].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    axes[0, col].set_title(f"{sample_id}\nnew: {band_name}")
    axes[1, col].imshow(reference, cmap="gray", vmin=vmin, vmax=vmax)
    axes[1, col].set_title(f"reference: {reference_band_name}")
    axes[2, col].imshow(instances, cmap="tab20", vmin=-0.5, vmax=19.5)
    axes[2, col].set_title(f"label | missing: {missing or 'none'}")
    axes[3, col].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    axes[3, col].imshow(instances, cmap="tab20", vmin=-0.5, vmax=19.5, alpha=0.45)
    axes[3, col].set_title("new chip + label")

for ax in axes.flat:
    ax.axis("off")
plt.show()